# LIBS Instrument Calibration Report

**Instrument:** <INSTRUMENT_NAME>

This report summarizes the instrument profile calibration process. It compares the measured experimental spectra with a synthetic spectrum generated using a two-zone plasma model (Hot Core + Cool Periphery).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Print timestamp of report generation
print(f"Report generated on: {time.strftime('%d-%m-%Y %H:%M:%S')}")

In [ ]:
inputCSV = "<INPUT_CSV_PATH>"
input_df = pd.read_csv(inputCSV, header=0, delimiter=';')
wavelengths_as_strings = input_df.columns.to_list()[2:]
input_wavelengths_list = list(map(float, wavelengths_as_strings))

measured_avg_df = pd.read_csv('target_processed.csv', header=0)
wavelengths_raw = measured_avg_df.iloc[:, 0].tolist()
measured_avg_intensities_raw = measured_avg_df.iloc[:, 1].tolist()

# Trimming original measured spectra for representation
startWavelength, endWavelength = wavelengths_raw[0], wavelengths_raw[-1]
startIndex, endIndex = input_wavelengths_list.index(startWavelength), input_wavelengths_list.index(endWavelength)+1
input_wavelengths_list = input_wavelengths_list[startIndex:endIndex]
input_wavelengths = np.array(input_wavelengths_list)

hot_df = pd.read_csv('hot/best_hot.csv', header=0)
hot_wavelengths_raw = hot_df.iloc[:, 0].tolist()
hot_intensities_raw = hot_df.iloc[:, 1].tolist()

cool_df = pd.read_csv('cool/best_cool.csv', header=0)
cool_wavelengths_raw = cool_df.iloc[:, 0].tolist()
cool_intensities_raw = cool_df.iloc[:, 1].tolist()

In [ ]:
# Interpolate the hot and cool spectra to match the measured wavelengths
hot_intensities_interp = np.interp(wavelengths_raw, hot_wavelengths_raw, hot_intensities_raw).tolist()
cool_intensities_interp = np.interp(wavelengths_raw, cool_wavelengths_raw, cool_intensities_raw).tolist()

In [ ]:
# --- DATA INJECTION SECTION ---
# The following variables are injected by the LIBS Data Curator tool

instrument_name = "<INSTRUMENT_NAME>"

# Wavelengths (nm)
wavelengths = np.array(wavelengths_raw)

# Measured Spectrum (Averaged)
measured_intensity = np.array(measured_avg_intensities_raw)

# Synthetic Components
hot_core_intensity = np.array(hot_intensities_interp)
cool_periphery_intensity = np.array(cool_intensities_interp)

# Parameters
hot_te = <HOT_CORE_TE>
hot_ne = <HOT_CORE_NE>
hot_weight = <WEIGHT>

cool_te = <COOL_PERIPHERY_TE>
cool_ne = <COOL_PERIPHERY_NE>
cool_weight = 1 - <WEIGHT>

# Calculated Combined Spectrum
combined_intensity = (hot_weight * hot_core_intensity) + (cool_weight * cool_periphery_intensity)

# Scores
rmse = <RMSE>
fit_score = <FIT_SCORE>

print(f"Calibration Report for: {instrument_name}")
print(f"Optimized Parameters:")
print(f"  Hot Core: Te={hot_te} eV, Ne={hot_ne:e} cm^-3")
print(f"  Cool Periphery: Te={cool_te} eV, Ne={cool_ne:e} cm^-3")
print(f"  Weight (Hot Core): {hot_weight:.4f}")
print(f"R^2: {fit_score:.4f}")
print(f"RMSE: {rmse:.4f}")

## 1. Experimental Data Visualization
### 1.1 Raw Measured Spectra (All Shots)

In [ ]:
plt.figure(figsize=(12, 6))
plt.title(f"Raw Measured Spectra - {instrument_name}")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Intensity (a.u.)")

# Plot all shots (limit to first 50 if too many to avoid clutter)
for index, row in input_df.iterrows():
    if index > 50: break
    plt.plot(input_wavelengths, row[startIndex+2:endIndex+2], alpha=0.6, linewidth=0.5)

plt.tight_layout()
plt.show()

### 1.2 Preprocessed Spectrum (Averaged & Baseline Corrected)
This spectrum serves as the target for the calibration optimization.

Baseline correction applied using Asymmetric Least Squares (ALS) with parameters:
* lambda = `<LAMBDA>`
* p = `<P>`  
* maxIterations = `<MAX_ITERATIONS>`

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(wavelengths, measured_intensity, label='Measured (Avg)', color='black', alpha=0.7, linewidth=1)

plt.title(f'Averaged measured spectrum')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Intensity (a.u.)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Synthetic Zone Spectra
Individual spectra generated for the optimized Hot Core and Cool Periphery parameters.

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(wavelengths, hot_core_intensity, label=f'Hot Core (Te={hot_te}eV, Ne={hot_ne:.1e})', color='orange')
plt.plot(wavelengths, cool_periphery_intensity, label=f'Cool Periphery (Te={cool_te}eV, Ne={cool_ne:.1e})', color='blue', alpha=0.6)

plt.title(f'Plasma Zone Contributions (Weight: Hot={hot_weight:.2f}, Cool={cool_weight:.2f})')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Intensity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Calibration Result & Validation
Overlay of the measured spectrum with the final weighted synthetic spectrum.

**Formula:** $I_{total} = w \cdot I_{hot} + (1-w) \cdot I_{cool}$

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(wavelengths, measured_intensity, label='Measured (Avg)', color='black', alpha=0.7, linewidth=1)
plt.plot(wavelengths, combined_intensity, label='Synthetic Two-Zone Spectrum', color='red', alpha=0.8, linestyle='--')

plt.title(f'Calibration Fit: Measured vs Synthetic (RMSE: {rmse:.4f}, R^2: {fit_score:.4f})')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Intensity (a.u.)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Residuals Plot

In [ ]:
# Residuals Plot
residuals = measured_intensity - combined_intensity
plt.figure(figsize=(15, 4))
plt.plot(wavelengths, residuals, color='green', alpha=0.7)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Residuals (Measured - Synthetic)')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Difference')
plt.grid(True, alpha=0.3)
plt.show()